# Signal normalisation

This notebook demonstrates two ways to normalise peak intensities across samples:

| Method          | Description                                                                               |
| --------------- | ----------------------------------------------------------------------------------------- |
| **TIC**         | Divide each peak's height by the Total Ion Count (sum of all peak heights) in that sample |
| **Reagent-ion** | Divide each peak's height by the reagent-ion intensity from a paired RI-acquisition batch |

Both methods produce a dimensionless ratio that removes sample-to-sample variation
in overall signal strength, making it easier to compare analyte trends.


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

from mascope_sdk import MascopeClient


pio.templates.default = "plotly_dark"  # or "plotly_white"

mascope = MascopeClient(workspace="My Workspace")

---

## Method 1: TIC normalisation

The **Total Ion Count** is the sum of all peak heights in a sample. Dividing each peak by the TIC gives a fractional contribution that is independent of overall signal level.


In [ ]:
# Load peaks (adjust dataset/batches to your data)
peaks = mascope.load_peaks(dataset="My Dataset", batches="My Batch")
peaks

In [ ]:
# Compute TIC per sample (sum of unique peak heights, before match expansion)
unique_peaks = peaks.drop_duplicates(subset=["sample_item_id", "peak_id"])

tic = unique_peaks.groupby("sample_item_id")["height"].sum().rename("tic")

# Join TIC back and compute normalised height
peaks_tic = peaks.merge(tic, on="sample_item_id")
peaks_tic["height_norm"] = peaks_tic["height"] / peaks_tic["tic"]

print(
    f"TIC range: [{tic.min():.2e} - {tic.max():.2e}], "
    f"mean {tic.mean():.2e}, std {tic.std():.2e} cps"
)
peaks_tic[["sample_item_name", "mz", "height", "tic", "height_norm"]].head()

In [ ]:
# Plot TIC across samples
tic_ts = (
    unique_peaks.drop_duplicates(subset=["sample_item_id"])[
        ["sample_item_id", "sample_item_name", "datetime_utc", "sample_batch_name"]
    ]
    .merge(tic, on="sample_item_id")
    .sort_values("datetime_utc")
)

fig = px.scatter(
    tic_ts,
    x="datetime_utc",
    y="tic",
    hover_data=["sample_item_name", "sample_batch_name"],
    title="Total Ion Count per sample",
)
fig.update_layout(
    xaxis_title="Datetime (UTC)", yaxis_title="TIC [cps]", yaxis_type="log"
)
fig.show()

In [ ]:
# Compare raw vs TIC-normalised timeseries for matched ions
matched_tic = peaks_tic[peaks_tic["target_ion_formula"].notna()].copy()

# Composite label: compound + ionization path
matched_tic["ion_formula_path"] = (
    matched_tic["target_compound_formula"] + matched_tic["ionization_mechanism"]
)

# Optional: filter to specific ions
# matched_tic = matched_tic[matched_tic["target_ion_formula"].isin(["C3H7O+"])]

ion_ts = (
    matched_tic.groupby(
        [
            "datetime_utc",
            "sample_item_name",
            "ion_formula_path",
            "target_ion_formula",
        ]
    )[["height", "height_norm"]]
    .sum()
    .reset_index()
    .sort_values("datetime_utc")
)

fig = px.scatter(
    ion_ts,
    x="datetime_utc",
    y="height",
    color="ion_formula_path",
    hover_data=["sample_item_name", "target_ion_formula"],
    title="Ion intensities: raw",
)
fig.update_layout(
    xaxis_title="Datetime (UTC)", yaxis_title="Intensity [cps]", yaxis_type="log"
)
fig.show()

fig = px.scatter(
    ion_ts,
    x="datetime_utc",
    y="height_norm",
    color="ion_formula_path",
    hover_data=["sample_item_name", "target_ion_formula"],
    title="Ion intensities: TIC normalised",
)
fig.update_layout(
    xaxis_title="Datetime (UTC)", yaxis_title="Height / TIC", yaxis_type="log"
)
fig.show()

---

## Method 2: Reagent ion normalisation

When data is acquired with two m/z ranges; one covering the **reagent ions** (RI) and one excluding them (No RI) for better dynamic range, we can normalise the analyte signals from the "No RI" batch by the reagent ion intensity from the paired "RI" batch.

Samples are matched **by position** (1st RI ↔ 1st No RI, etc.), so both batches must have the same number of samples in the same acquisition order.

If multiple reagent ions are specified, their intensities are **summed** per sample.


In [ ]:
# --- Configuration ---
dataset_name = "My Dataset"

ri_batch = "Uronium RI"  # batches including reagent ion
no_ri_batch = "Uronium No RI"  # batches to normalize

# Reagent ion(s) to normalise by, identified by ion formula
reagent_ion_formulas = [
    "CH5N2O+",
    "C2H9N4O2+",
]  # configure to match your data

In [ ]:
# Load both batches
peaks_ri = mascope.load_peaks(dataset=dataset_name, batches=ri_batch)
peaks_no_ri = mascope.load_peaks(dataset=dataset_name, batches=no_ri_batch)

print(f"RI batches:    {peaks_ri['sample_item_name'].nunique()} samples")
print(f"No RI batches: {peaks_no_ri['sample_item_name'].nunique()} samples")

In [ ]:
# Extract reagent-ion intensity per RI sample
ri_matched = peaks_ri[peaks_ri["target_ion_formula"].isin(reagent_ion_formulas)].copy()

if ri_matched.empty:
    raise ValueError(
        f"No peaks matched to reagent ions {reagent_ion_formulas} in the RI batch. "
        "Check the ion formulas or batch name."
    )

# Sum reagent-ion heights per sample (handles multiple reagent ions)
ri_intensity = (
    ri_matched.drop_duplicates(subset=["sample_item_id", "peak_id"])
    .groupby("sample_item_id")["height"]
    .sum()
    .rename("ri_height")
)

print(
    f"Reagent ion intensity range: [{ri_intensity.min():.2e} – {ri_intensity.max():.2e}], "
    f"mean {ri_intensity.mean():.2e}, std {ri_intensity.std():.2e} cps"
)
print(
    f"Found in {len(ri_intensity)} / {peaks_ri['sample_item_id'].nunique()} RI samples"
)

In [ ]:
# Plot reagent ion intensity across samples

ri_ts = (
    ri_intensity.reset_index()
    .merge(
        peaks_ri[
            ["sample_item_id", "sample_item_name", "datetime_utc", "sample_batch_name"]
        ].drop_duplicates(),
        on="sample_item_id",
    )
    .sort_values("datetime_utc")
)

fig = px.scatter(
    ri_ts,
    x="datetime_utc",
    y="ri_height",
    hover_data=["sample_item_name", "sample_batch_name"],
    title=f"Reagent Ion ({', '.join(reagent_ion_formulas)}) Intensity per sample",
)
fig.update_layout(
    xaxis_title="Datetime (UTC)", yaxis_title="RI Intensity [cps]", yaxis_type="log"
)
fig.show()

In [ ]:
# Match RI and No-RI samples by position (acquisition order)
ri_samples = (
    peaks_ri[["sample_item_id", "sample_item_name", "datetime_utc"]]
    .drop_duplicates()
    .sort_values("datetime_utc")
    .reset_index(drop=True)
)
no_ri_samples = (
    peaks_no_ri[["sample_item_id", "sample_item_name", "datetime_utc"]]
    .drop_duplicates()
    .sort_values("datetime_utc")
    .reset_index(drop=True)
)

if len(ri_samples) != len(no_ri_samples):
    print(
        f"⚠ Sample count mismatch: {len(ri_samples)} RI vs {len(no_ri_samples)} No RI. "
        "Pairing as many as possible. Check the result and adjust offset if needed."
    )

# Adjust offsets if RI samples are not perfectly aligned with No-RI samples.
# Add offset to the batch with more samples to skip extras.
ri_offset = 0
no_ri_offset = 0

n_pairs = min(len(ri_samples) - ri_offset, len(no_ri_samples) - no_ri_offset)
sample_pairs = pd.DataFrame(
    {
        "ri_sample_id": ri_samples["sample_item_id"].values[
            ri_offset : n_pairs + ri_offset
        ],
        "ri_sample_name": ri_samples["sample_item_name"].values[
            ri_offset : n_pairs + ri_offset
        ],
        "no_ri_sample_id": no_ri_samples["sample_item_id"].values[
            no_ri_offset : n_pairs + no_ri_offset
        ],
        "no_ri_sample_name": no_ri_samples["sample_item_name"].values[
            no_ri_offset : n_pairs + no_ri_offset
        ],
    }
)

# Attach RI intensity to each pair
sample_pairs = sample_pairs.merge(
    ri_intensity, left_on="ri_sample_id", right_index=True, how="left"
)

missing_ri = sample_pairs["ri_height"].isna().sum()
if missing_ri:
    print(
        f"⚠ {missing_ri} paired RI samples have no reagent-ion signal, these will not be normalised."
    )

print(f"{n_pairs} sample pairs:")
sample_pairs

In [ ]:
# Join RI intensity onto No-RI peaks and normalise
ri_map = sample_pairs.set_index("no_ri_sample_id")["ri_height"]

peaks_norm = peaks_no_ri.copy()
peaks_norm["ri_height"] = peaks_norm["sample_item_id"].map(ri_map)
peaks_norm["height_norm"] = peaks_norm["height"] / peaks_norm["ri_height"]

peaks_norm[["sample_item_name", "mz", "height", "ri_height", "height_norm"]]

In [ ]:
# Compare raw vs reagent-ion-normalised timeseries
matched_norm = peaks_norm[peaks_norm["target_ion_formula"].notna()].copy()

matched_norm["ion_formula_path"] = (
    matched_norm["target_compound_formula"] + matched_norm["ionization_mechanism"]
)

# Optional: filter to specific ions
# matched_norm = matched_norm[matched_norm["target_ion_formula"].isin(["C3H7O+"])]

ion_ts = (
    matched_norm.groupby(
        [
            "datetime_utc",
            "sample_item_name",
            "ion_formula_path",
            "target_ion_formula",
        ]
    )[["height", "height_norm"]]
    .sum()
    .reset_index()
    .sort_values("datetime_utc")
)

fig = px.scatter(
    ion_ts,
    x="datetime_utc",
    y="height",
    color="ion_formula_path",
    hover_data=["sample_item_name", "target_ion_formula"],
    title="Ion intensities: raw",
)
fig.update_layout(
    xaxis_title="Datetime (UTC)", yaxis_title="Intensity [cps]", yaxis_type="log"
)
fig.show()

fig = px.scatter(
    ion_ts,
    x="datetime_utc",
    y="height_norm",
    color="ion_formula_path",
    hover_data=["sample_item_name", "target_ion_formula"],
    title="Ion intensities: normalised by reagent ion",
)
fig.update_layout(
    xaxis_title="Datetime (UTC)", yaxis_title="Height / RI intensity", yaxis_type="log"
)
fig.show()